# Case Study 2: Hierarchical Linear Model (HLM)

**Circulatory Fidelity v1.1: Inference Coupling Diagnostic**

This notebook demonstrates how IC diagnoses when hierarchical structure is essential vs. redundant in pooling models.

---

## Key Concepts

- **IC = √ICC**: For HLMs, IC equals the square root of the Intraclass Correlation Coefficient
- **Pooling model interpretation**: High IC → signal reliable → hierarchy redundant
- **Low IC → signal unreliable → pooling essential**

---

## Model Specification

Standard hierarchical linear model:

$$
\begin{align}
\mu &\sim \mathcal{N}(0, \sigma_\mu^2) \quad \text{[grand mean]}\\
\theta_j | \mu &\sim \mathcal{N}(\mu, \tau^2) \quad \text{[group effects]}\\
y_{ij} | \theta_j &\sim \mathcal{N}(\theta_j, \sigma^2) \quad \text{[observations]}
\end{align}
$$

The **Intraclass Correlation Coefficient**:
$$\text{ICC} = \frac{\tau^2}{\tau^2 + \sigma^2}$$

And **Inference Coupling**:
$$\IC = \sqrt{\text{ICC}}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from dataclasses import dataclass
from typing import Tuple, List

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
np.random.seed(42)

## Core Functions

In [ ]:
def icc_from_variances(tau_sq: float, sigma_sq: float) -> float:
    """Compute ICC from variance components."""
    return tau_sq / (tau_sq + sigma_sq)

def ic_from_icc(icc: float) -> float:
    """Compute IC from ICC: IC = √ICC"""
    return np.sqrt(max(0, icc))

def reliability(icc: float, n_per_group: int) -> float:
    """
    Spearman-Brown reliability: how well group means estimate true effects.
    
    rel = n × ICC / (1 + (n-1) × ICC)
    """
    return (n_per_group * icc) / (1 + (n_per_group - 1) * icc)

## HLM Simulation

In [ ]:
@dataclass
class HLMParams:
    """HLM parameters."""
    n_groups: int = 20
    n_per_group: int = 10
    tau: float = 1.0        # Between-group SD
    sigma: float = 1.0      # Within-group SD
    grand_mean: float = 0.0
    
    @property
    def icc(self) -> float:
        return icc_from_variances(self.tau**2, self.sigma**2)
    
    @property
    def ic(self) -> float:
        return ic_from_icc(self.icc)

def simulate_hlm(params: HLMParams, seed: int = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate HLM data."""
    if seed is not None:
        np.random.seed(seed)
    
    # Group effects
    theta = np.random.normal(params.grand_mean, params.tau, params.n_groups)
    
    # Observations
    y = []
    groups = []
    for j in range(params.n_groups):
        y_j = np.random.normal(theta[j], params.sigma, params.n_per_group)
        y.extend(y_j)
        groups.extend([j] * params.n_per_group)
    
    return np.array(y), np.array(groups), theta

## Estimation Methods

In [ ]:
def no_pooling_estimate(y: np.ndarray, groups: np.ndarray, n_groups: int) -> np.ndarray:
    """No-pooling: each group mean estimated independently."""
    theta_hat = np.zeros(n_groups)
    for j in range(n_groups):
        theta_hat[j] = np.mean(y[groups == j])
    return theta_hat

def partial_pooling_estimate(y: np.ndarray, groups: np.ndarray, 
                            n_groups: int, icc: float, n_per_group: int) -> np.ndarray:
    """Partial pooling: shrink toward grand mean."""
    grand_mean = np.mean(y)
    group_means = no_pooling_estimate(y, groups, n_groups)
    
    # Shrinkage factor
    rel = reliability(icc, n_per_group)
    
    # Shrink toward grand mean
    theta_hat = rel * group_means + (1 - rel) * grand_mean
    return theta_hat

## Single Example: High vs Low ICC

In [ ]:
# Low ICC (IC ≈ 0.3) - pooling essential
params_low = HLMParams(tau=0.3, sigma=1.0)
print(f"Low signal scenario:")
print(f"  τ = {params_low.tau}, σ = {params_low.sigma}")
print(f"  ICC = {params_low.icc:.3f}")
print(f"  IC = {params_low.ic:.3f}")

# High ICC (IC ≈ 0.9) - pooling redundant
params_high = HLMParams(tau=3.0, sigma=1.0)
print(f"\nHigh signal scenario:")
print(f"  τ = {params_high.tau}, σ = {params_high.sigma}")
print(f"  ICC = {params_high.icc:.3f}")
print(f"  IC = {params_high.ic:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, params, title in [(axes[0], params_low, 'Low IC: Pooling Essential'),
                          (axes[1], params_high, 'High IC: Pooling Redundant')]:
    y, groups, theta_true = simulate_hlm(params, seed=42)
    
    theta_np = no_pooling_estimate(y, groups, params.n_groups)
    theta_pp = partial_pooling_estimate(y, groups, params.n_groups, 
                                       params.icc, params.n_per_group)
    
    mse_np = np.mean((theta_np - theta_true)**2)
    mse_pp = np.mean((theta_pp - theta_true)**2)
    
    x = np.arange(params.n_groups)
    width = 0.25
    
    ax.bar(x - width, theta_true, width, label='True θ', color='black', alpha=0.7)
    ax.bar(x, theta_np, width, label=f'No-pool (MSE={mse_np:.3f})', color='red', alpha=0.7)
    ax.bar(x + width, theta_pp, width, label=f'Partial-pool (MSE={mse_pp:.3f})', color='blue', alpha=0.7)
    
    ax.set_xlabel('Group')
    ax.set_ylabel('Effect estimate')
    ax.set_title(f'{title}\nIC = {params.ic:.2f}, MSE ratio = {mse_np/mse_pp:.1f}×')
    ax.legend(fontsize=9)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Validation Against Manuscript Data

In [ ]:
# Load manuscript validation data
try:
    df = pd.read_csv('../data/hlm_validation.csv')
    print(f"Loaded validation data: {len(df)} simulations")
    print(f"\nSummary statistics:")
    print(f"  IC range: {df['ic'].min():.4f} - {df['ic'].max():.4f}")
    print(f"  ICC range: {df['icc'].min():.4f} - {df['icc'].max():.4f}")
    print(f"  MSE ratio range: {df['mse_ratio'].min():.2f} - {df['mse_ratio'].max():.2f}")
    
    # Verify IC = sqrt(ICC)
    df['ic_computed'] = np.sqrt(df['icc'])
    diff = np.abs(df['ic'] - df['ic_computed']).max()
    print(f"\nVerification: max|IC - √ICC| = {diff:.6f}")
    
except FileNotFoundError:
    print("Validation data not found.")
    df = None

In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    # Panel A: IC vs ICC (should be perfect sqrt relationship)
    grouped = df.groupby('tau').agg({'ic': 'mean', 'icc': 'mean'}).reset_index()
    axes[0].scatter(grouped['icc'], grouped['ic'], s=80, c='black')
    icc_line = np.linspace(0, grouped['icc'].max(), 100)
    axes[0].plot(icc_line, np.sqrt(icc_line), 'r--', label='IC = √ICC')
    axes[0].set_xlabel('ICC')
    axes[0].set_ylabel('IC')
    axes[0].set_title('(A) IC = √ICC (by construction)')
    axes[0].legend()
    
    # Panel B: MSE Ratio vs ICC
    grouped_mse = df.groupby('tau').agg({'mse_ratio': ['mean', 'std'], 'icc': 'mean'}).reset_index()
    grouped_mse.columns = ['tau', 'mse_mean', 'mse_std', 'icc']
    axes[1].errorbar(grouped_mse['icc'], grouped_mse['mse_mean'], yerr=grouped_mse['mse_std'],
                    fmt='s-', color='black', capsize=3)
    axes[1].axhline(2.0, color='red', linestyle='--', label='Threshold (2×)')
    axes[1].set_xlabel('ICC')
    axes[1].set_ylabel('MSE Ratio (No-pool / Partial-pool)')
    axes[1].set_title('(B) Pooling benefit decreases with ICC')
    axes[1].legend()
    
    # Panel C: IC vs MSE Ratio (negative correlation)
    subsample = df.sample(n=min(500, len(df)), random_state=42)
    axes[2].scatter(subsample['ic'], subsample['mse_ratio'], alpha=0.3, s=15, c='gray')
    r = np.corrcoef(df['ic'], df['mse_ratio'])[0, 1]
    axes[2].set_xlabel('IC')
    axes[2].set_ylabel('MSE Ratio')
    axes[2].set_title(f'(C) Low IC → pooling essential (r = {r:.2f})')
    
    plt.tight_layout()
    plt.show()

## The Dependency Asymmetry

**Key insight**: IC means opposite things in filtering vs. pooling models!

| Model Type | High IC Means | Low IC Means |
|------------|---------------|---------------|
| **Filtering** (SVF) | MFVI fails | MFVI OK |
| **Pooling** (HLM) | Signal reliable, hierarchy redundant | Signal unreliable, pooling essential |

This is the **Dependency Asymmetry** resolved by distinguishing:
- **Constitutive coupling** (filtering): Dependencies are load-bearing
- **Inductive coupling** (pooling): Dependencies are scaffolding

## Practical Recommendation (Pooling Models)

**Interpretive Scale** (from manuscript Section 2.7):

For pooling models, the interpretation inverts—low IC indicates groups are similar (strong pooling needed):

| IC Range | Coupling Regime | Recommendation |
|----------|-----------------|----------------|
| < 0.25 | Negligible | Strong pooling required (groups very similar) |
| 0.25 - 0.35 | Weak | Strong pooling required |
| 0.35 - 0.55 | Moderate | Partial pooling recommended |
| 0.55 - 0.70 | Strong | Partial pooling optional |
| > 0.70 | Very strong | No-pooling acceptable (groups distinct) |

---

*Notebook aligned with Circulatory Fidelity v1.1 manuscript*
